In [0]:
%pip install -q databricks-sdk>=0.118.0 "psycopg[binary]>=3.1.0"
dbutils.library.restartPython()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googleapis-common-protos 1.65.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0.dev0,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.67.0 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Milestone 2.9 — Low-Latency Business Domain Query (Direct Lakebase Connection)
# Connects directly to Lakebase Postgres via psycopg2 using OAuth credentials
# from the Databricks SDK — no Spark/Lakehouse intermediary.

from databricks.sdk import WorkspaceClient
import psycopg
import pandas as pd


In [0]:
# Lakebase connection parameters ---
PROJECT = "meridian-bank"
BRANCH = "production"
ENDPOINT = "primary"
DATABASE = "databricks_postgres"
HOST = "ep-morning-art-e1k1x4aq.database.eastus2.azuredatabricks.net"

# Generate short-lived OAuth credential via Databricks SDK
w = WorkspaceClient()
username = w.current_user.me().user_name
token = w.postgres.generate_database_credential(
    endpoint=f"projects/{PROJECT}/branches/{BRANCH}/endpoints/{ENDPOINT}"
).token


In [0]:
# Connect directly to Lakebase Postgres ---
conn = psycopg.connect(
    host=HOST,
    port=5432,
    dbname=DATABASE,
    user=username,
    password=token,
    sslmode="require",
)


In [0]:
# Business Query: High-value at-risk customers with NBA recommendations ---
query = """
SELECT
    cp.customer_id,
    cp.customer_display_name,
    cp.deposit_balance_usd,
    cp.tenure_years,
    cp.tier,
    ar.atrisk_product_id,
    ar.attrition_risk_score,
    ar.days_to_maturity,
    nba.recommended_action,
    nba.predicted_retained_usd,
    p.product_name AS recommended_product,
    p.rate_apy AS offer_rate
FROM meridian_bank.synced_gold_customer_position cp
JOIN meridian_bank.synced_gold_open_atrisk ar
    ON cp.customer_id = ar.customer_id
JOIN meridian_bank.synced_gold_nba_recommendations nba
    ON cp.customer_id = nba.customer_id
LEFT JOIN meridian_bank.products p
    ON nba.recommended_offer_product_id = p.product_id
WHERE cp.deposit_balance_usd > 100000
  AND ar.attrition_risk_score > 0.6
ORDER BY ar.attrition_risk_score DESC, cp.deposit_balance_usd DESC
LIMIT 10;
"""

with conn.cursor() as cur:
    cur.execute(query)
    columns = [desc[0] for desc in cur.description]
    rows = cur.fetchall()

conn.close()

df = pd.DataFrame(rows, columns=columns)
print(f"✓ Returned {len(df)} high-risk customers from Lakebase (direct Postgres connection)")
print(df.to_string(index=False))


✓ Returned 10 high-risk customers from Lakebase (direct Postgres connection)
 customer_id customer_display_name  deposit_balance_usd  tenure_years     tier atrisk_product_id  attrition_risk_score  days_to_maturity recommended_action  predicted_retained_usd recommended_product offer_rate
CUST-0004138      Customer 0004138           1249678.51            19 affluent     PROD-DEP-2003                  0.95                26    retention_offer            77545.848390                None       None
CUST-0001326      Customer 0001326           1210046.73            13 affluent     PROD-DEP-2003                  0.95                 7    retention_offer            73513.893480                None       None
CUST-0005063      Customer 0005063           1209662.12             8 affluent     PROD-DEP-2001                  0.95                39    retention_offer            73677.233123                None       None
CUST-0006136      Customer 0006136           1178486.53            18 affluent 